# Prepare VI datasets from spin-up NPZ

Build VI-ready bundles from production spin-up `_grid.npz` ground truth (more sliding / no sliding).

Each bundle adds masks, log-viscosity targets, and synthetic noisy velocity observations, then writes:

- `outputs/vi/vi_case_<case_id>.npz`
- `outputs/vi/manifest.json`

Source paths come from `scripts/project_paths.py` (`PRODUCTION_CASES`). Core logic lives in `scripts/prep_vi_dataset.py`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "scripts" / "project_paths.py").exists()
)
sys.path.insert(0, str(ROOT / "scripts"))

from project_paths import PRODUCTION_CASES, VI_DIR
from prep_vi_dataset import (
    NoiseConfig,
    build_manifest,
    compute_normalization,
    prepare_case,
    read_jsonable,
)

## Configuration

In [2]:
OUTPUT_DIR = VI_DIR
NOISE_SIGMA = 0.02  # relative Gaussian noise on speed (2%)
SEED = 42

CASE_SPECS = {
    "more_sliding": {
        "source_npz": PRODUCTION_CASES["more_sliding"]["grid_npz"],
        "sliding_regime": "high",
    },
    "no_sliding": {
        "source_npz": PRODUCTION_CASES["no_sliding"]["grid_npz"],
        "sliding_regime": "low",
    },
}

noise = NoiseConfig(relative_speed_sigma=NOISE_SIGMA, seed=SEED)
output_dir = OUTPUT_DIR.resolve()
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir.relative_to(ROOT)}")
print(f"Noise: relative_speed_sigma={noise.relative_speed_sigma}, seed={noise.seed}")
print()
for case_id, spec in CASE_SPECS.items():
    path = spec["source_npz"]
    status = "✓" if path.is_file() else "MISSING"
    print(f"{case_id}: {path.relative_to(ROOT)} {status}")

Output directory: outputs/vi
Noise: relative_speed_sigma=0.02, seed=42

more_sliding: outputs/spinup/production/more_sliding/SteadyState_more_sliding_10500yr_ramp4000_1refine_grid.npz ✓
no_sliding: outputs/spinup/production/no_sliding/SteadyState_no_sliding_10500yr_ramp4000_1refine_grid.npz ✓


## Build VI bundles

In [3]:
rng = np.random.default_rng(noise.seed)
entries = []
prepared_fields = {}

for case_id, spec in CASE_SPECS.items():
    entry, fields = prepare_case(
        case_id,
        spec["source_npz"].resolve(),
        spec["sliding_regime"],
        output_dir,
        noise,
        rng,
    )
    entries.append(entry)
    prepared_fields[case_id] = fields
    print(f"Wrote {Path(entry['vi_bundle']).relative_to(ROOT)}")

normalization = compute_normalization(prepared_fields)
manifest = build_manifest(entries, output_dir, noise, normalization)
manifest_path = output_dir / "manifest.json"
with manifest_path.open("w", encoding="utf-8") as stream:
    json.dump(read_jsonable(manifest), stream, indent=2, sort_keys=True)

print(f"\nWrote {manifest_path.relative_to(ROOT)}")
print(f"Prepared {len(entries)} VI cases")

Wrote outputs/vi/vi_case_more_sliding.npz
Wrote outputs/vi/vi_case_no_sliding.npz

Wrote outputs/vi/manifest.json
Prepared 2 VI cases


## Manifest summary

In [4]:
for entry in entries:
    print(f"=== {entry['case_id']} ({entry['sliding_regime']} sliding) ===")
    print(f"  source: {Path(entry['source_npz']).relative_to(ROOT)}")
    print(f"  bundle: {Path(entry['vi_bundle']).relative_to(ROOT)}")
    print(f"  C = {entry['C']} | grounded = {entry['fraction_grounded']:.1%}")
    for name, stats in entry["summary"].items():
        print(
            f"  {name}: mean={stats['mean']:.4g}, "
            f"std={stats['std']:.4g}, max={stats['max']:.4g}"
        )
    print()

print("Normalization (pooled over grounded cells):")
for field, stats in normalization.items():
    print(f"  {field}: {stats}")

=== more_sliding (high sliding) ===
  source: outputs/spinup/production/more_sliding/SteadyState_more_sliding_10500yr_ramp4000_1refine_grid.npz
  bundle: outputs/vi/vi_case_more_sliding.npz
  C = 0.001 | grounded = 87.1%
  speed_obs: mean=193.4, std=237.2, max=1047
  viscosity_true: mean=16.02, std=9.318, max=42.55
  log_viscosity_true: mean=2.585, std=0.6468, max=3.751

=== no_sliding (low sliding) ===
  source: outputs/spinup/production/no_sliding/SteadyState_no_sliding_10500yr_ramp4000_1refine_grid.npz
  bundle: outputs/vi/vi_case_no_sliding.npz
  C = 100.0 | grounded = 88.9%
  speed_obs: mean=146.2, std=254.4, max=1064
  viscosity_true: mean=28.27, std=17.06, max=60.85
  log_viscosity_true: mean=3.068, std=0.8447, max=4.108

Normalization (pooled over grounded cells):
  speed_obs: {'mean': 169.55349572634748, 'std': 247.21565033452077}
  log_viscosity_true: {'mean': 2.8290880894361807, 'std': 0.7909941894994298}
  bed: {'min': -733.11634617052, 'max': 349.83232493477317}
